In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression

import xgboost as xgb
import lightgbm as lgb

from imblearn.over_sampling import SMOTE

import joblib
import warnings
warnings.filterwarnings("ignore")



In [ ]:
df = pd.read_csv("churn.csv")

df.drop("customerID", axis=1, inplace=True)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

df["Churn"] = (df["Churn"] == "Yes").astype(int)

In [ ]:
class ChurnPreprocessor:
    def __init__(self):
        self.binary_cols = ["gender","Partner","Dependents","PhoneService","PaperlessBilling"]

        self.cat_cols = [
            "MultipleLines","InternetService","OnlineSecurity",
            "OnlineBackup","DeviceProtection","TechSupport",
            "StreamingTV","StreamingMovies","Contract",
            "PaymentMethod"
        ]

        self.contract_map = {"Month-to-month":2, "One year":1, "Two year":0}
        self.feature_columns = None

    def feature_engineering(self, df):
        df = df.copy()

        df["ChargesPerMonth"] = df["TotalCharges"] / (df["tenure"] + 1)
        df["MonthlyToTotal"] = df["MonthlyCharges"] / (df["TotalCharges"] + 1)

        df["TenureBucket"] = pd.cut(
            df["tenure"],
            bins=[-1, 12, 36, 72],
            labels=[0, 1, 2]
        ).astype(int)

        service_cols = [
            "PhoneService","MultipleLines","InternetService",
            "OnlineSecurity","OnlineBackup","DeviceProtection",
            "TechSupport","StreamingTV","StreamingMovies"
        ]

        df["ServiceCount"] = df[service_cols].apply(
            lambda row: sum(v not in ["No","No internet service","No phone service"] for v in row),
            axis=1
        )

        df["HighValue"] = (df["MonthlyCharges"] > df["MonthlyCharges"].median()).astype(int)
        df["ContractRisk"] = df["Contract"].map(self.contract_map)

        return df

    def fit_transform(self, df):
        df = self.feature_engineering(df)

        # binary encoding (SAFE)
        for col in self.binary_cols:
            df[col] = df[col].map({"Male":0, "Female":1}) if col == "gender" else df[col].map({"No":0, "Yes":1})

        df = pd.get_dummies(df, columns=self.cat_cols, drop_first=True)
        df = df.replace({True:1, False:0})

        self.feature_columns = df.drop("Churn", axis=1).columns.tolist()

        return df

    def transform(self, df):
        df = self.feature_engineering(df)

        for col in self.binary_cols:
            df[col] = df[col].map({"Male":0, "Female":1}) if col == "gender" else df[col].map({"No":0, "Yes":1})

        df = pd.get_dummies(df, columns=self.cat_cols, drop_first=True)
        df = df.replace({True:1, False:0})

        # align columns
        for col in self.feature_columns:
            if col not in df.columns:
                df[col] = 0

        return df[self.feature_columns]


In [ ]:
preprocessor = ChurnPreprocessor()
df = preprocessor.fit_transform(df)

X = df.drop("Churn", axis=1)
y = df["Churn"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
smote = SMOTE(random_state=42)
scaler = StandardScaler()

X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

X_train_sc = scaler.fit_transform(X_train_res)
X_test_sc = scaler.transform(X_test)


In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=40,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

gb_model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    random_state=42
)

rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)



In [ ]:
stack = StackingClassifier(
    estimators=[
        ("xgb", xgb_model),
        ("lgb", lgb_model),
        ("gb", gb_model),
        ("rf", rf_model)
    ],
    final_estimator=LogisticRegression(max_iter=500),
    cv=5,
    stack_method="predict_proba",
    n_jobs=-1
)


In [ ]:
stack.fit(X_train_sc, y_train_res)

y_proba = stack.predict_proba(X_test_sc)[:, 1]
y_pred = (y_proba >= 0.35).astype(int)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


Accuracy: 0.7615330021291696
ROC-AUC: 0.8269239711694955
              precision    recall  f1-score   support

           0       0.88      0.79      0.83      1035
           1       0.54      0.70      0.61       374

    accuracy                           0.76      1409
   macro avg       0.71      0.74      0.72      1409
weighted avg       0.79      0.76      0.77      1409

[[813 222]
 [114 260]]


In [ ]:
joblib.dump(stack, "ensemble.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")
joblib.dump(list(X.columns), "feature_columns.pkl")

print("\n✅ ALL ARTIFACTS SAVED SUCCESSFULLY")


✅ ALL ARTIFACTS SAVED SUCCESSFULLY
